In [1]:
# Import modules
import os
import sys

# Make sure that the executable path exists in the environment path
env_bin = os.path.dirname(sys.executable)
if env_bin not in os.environ["PATH"]:
    os.environ["PATH"] = f"{env_bin}:{os.environ['PATH']}"

import numpy as np
import xarray as xr
import yaml

with open("config.yml") as f:
    config = yaml.safe_load(f)
viz_opts = config["viz"]

In [2]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=4, threads_per_worker=4)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 16,Total memory: 128.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34549,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38319,Total threads: 4
Dashboard: http://127.0.0.1:45551/status,Memory: 32.00 GiB
Nanny: tcp://127.0.0.1:39741,


In [3]:
surface_velocity = xr.open_zarr("./surface_stress_ocean_velocity.zarr", chunks="auto")
surface_velocity

<xarray.Dataset> Size: 10MB
Dimensions:    (time: 151, latitude: 50, longitude: 161)
Coordinates:
  * time       (time) datetime64[ns] 1kB 2017-01-03 2017-01-08 ... 2019-01-23
  * latitude   (latitude) float64 400B -74.5 -74.0 -73.5 ... -51.0 -50.5 -50.0
  * longitude  (longitude) float64 1kB -90.0 -89.5 -89.0 ... -11.0 -10.5 -10.0
Data variables:
    u          (time, latitude, longitude) float32 5MB dask.array<chunksize=(151, 50, 161), meta=np.ndarray>
    v          (time, latitude, longitude) float32 5MB dask.array<chunksize=(151, 50, 161), meta=np.ndarray>
Attributes:
    CDI:          Climate Data Interface version 1.9.7.1 (http://mpimet.mpg.d...
    Conventions:  CF-1.6
    history:      Fri Oct 22 15:31:36 2021: cdo -f nc -b F32 copy ./mom5_regr...
    filename:     ./mom5_regrided.nc                   
    title:        MOM5_SODA_3.15.2
    grid_type:    mosaic
    grid_tile:    1
    CDO:          Climate Data Operators version 1.9.7.1 (http://mpimet.mpg.d...

In [4]:
wind = xr.open_zarr("./surface_stress_wind.zarr", chunks="auto")
wind = wind.drop_vars(["isobaricInhPa", "number", "step"])
wind

<xarray.Dataset> Size: 15MB
Dimensions:    (time: 220, latitude: 80, longitude: 110)
Coordinates:
  * time       (time) datetime64[ns] 2kB 2016-12-29 2017-01-03 ... 2019-12-29
  * latitude   (latitude) float64 640B -50.0 -50.5 -51.0 ... -88.5 -89.0 -89.5
  * longitude  (longitude) float64 880B -75.0 -74.5 -74.0 ... -21.5 -21.0 -20.5
Data variables:
    u          (time, latitude, longitude) float32 8MB dask.array<chunksize=(220, 80, 110), meta=np.ndarray>
    v          (time, latitude, longitude) float32 8MB dask.array<chunksize=(220, 80, 110), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-10-17T01:04 GRIB to CDM+CF via cfgrib-0.9.1...

In [5]:
sea_ice_cover = xr.open_zarr("./surface_stress_ice_cover.zarr", chunks="auto")
sea_ice_cover = sea_ice_cover.coarsen(latitude=2, longitude=2, boundary="trim").mean()
# coordinates are now block midpoints (x.25 -> x.125 etc.); snap to the half-degree grid
# introduces a 1/8 degree shift, but regridder crashes the kernel
sea_ice_cover = sea_ice_cover.assign_coords(
    latitude=np.round(sea_ice_cover.latitude * 2) / 2,
    longitude=np.round(sea_ice_cover.longitude * 2) / 2,
)
sea_ice_cover = sea_ice_cover.drop_vars(["lat", "lon", "x", "y", "band", "spatial_ref"])
sea_ice_cover

<xarray.Dataset> Size: 12MB
Dimensions:    (time: 220, latitude: 60, longitude: 110)
Coordinates:
  * time       (time) datetime64[ns] 2kB 2016-12-29 2017-01-03 ... 2019-12-29
  * latitude   (latitude) float64 480B -80.0 -79.5 -79.0 ... -51.5 -51.0 -50.5
  * longitude  (longitude) float64 880B -75.0 -74.5 -74.0 ... -21.5 -21.0 -20.5
Data variables:
    z          (time, latitude, longitude) float64 12MB dask.array<chunksize=(220, 60, 110), meta=np.ndarray>

In [6]:
sea_ice_drift = xr.open_zarr("surface_stress_sea_ice_drift.zarr", chunks="auto")
sea_ice_drift

<xarray.Dataset> Size: 457MB
Dimensions:    (time: 220, latitude: 361, longitude: 720)
Coordinates:
  * time       (time) datetime64[ns] 2kB 2016-12-29 2017-01-03 ... 2019-12-29
  * latitude   (latitude) float64 3kB -90.0 -89.5 -89.0 -88.5 ... 89.0 89.5 90.0
  * longitude  (longitude) float64 6kB -180.0 -179.5 -179.0 ... 179.0 179.5
Data variables:
    u          (time, latitude, longitude) float32 229MB dask.array<chunksize=(112, 361, 720), meta=np.ndarray>
    v          (time, latitude, longitude) float32 229MB dask.array<chunksize=(112, 361, 720), meta=np.ndarray>
Attributes: (12/30)
    Conventions:                CF-1.6
    title:                      Daily gridded sea ice motion from combined Bu...
    source:                     Tschudi, M., W. N. Meier, J. S. Steward, C. F...
    history:                    Modified from source (see above) on Mon Apr 2...
    cdm_data_type:              EASE grid
    geographic_region:          polar regions
    ...                         ...
    geospatial_lat_resolution:  25km
    platform:                   NOAA-Pathfinder NIMBUS DMSP
    sensor:                     AVHRR SMMR SSM/I SSMIS
    version:                    4.1
    citation:                   [1] Tschudi, M. A., Meier, W. N., and Stewart...
    regrid_method:              conservative

## Momentum transfer
Equations are taken from Ralphs PhD thesis. See also [Losch et al., 2012](https://doi.org/10.1016/j.ocemod.2011.10.002)
\begin{equation}
    \vec{\tau} = A \vec{\tau}_\text{io} + (1-A) \vec{\tau}_\text{ao}
\end{equation}
with atmosphere--ocean interaction $\vec{\tau}_\text{ao}$
\begin{equation}
|\vec{\tau}_\text{ao}| = \rho_\text{a} c_\text{D, ao} |u_{10}|^2 \qquad \text{with}\: \rho_\text{a} \approx 1.3\,\text{kg/m}^3,\:c_\text{D, ao} = 1\times10^{-3}
\end{equation}
and ice-ocean interaction $\vec{\tau}_\text{io}$
\begin{equation}
|\vec{\tau}_\text{io}| = \rho_\text{w} c_\text{D, io} |(\vec{u}_\text{o}-\vec{u}_\text{i})|^2 \qquad \text{with}\: \rho_\text{w} \approx 1000\,\text{kg/m}^3,\:c_\text{D, io} = 5.5\times10^{-3}
\end{equation}

In [7]:
tau_atmosphere_ocean = 1.3 * 1e-3 * np.sqrt(wind.u**2 + wind.v**2) * (wind.u + 1j * wind.v)
tau_atmosphere_ocean

<xarray.DataArray (time: 220, latitude: 80, longitude: 110)> Size: 15MB
dask.array<mul, shape=(220, 80, 110), dtype=complex64, chunksize=(220, 80, 110), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 2kB 2016-12-29 2017-01-03 ... 2019-12-29
  * latitude   (latitude) float64 640B -50.0 -50.5 -51.0 ... -88.5 -89.0 -89.5
  * longitude  (longitude) float64 880B -75.0 -74.5 -74.0 ... -21.5 -21.0 -20.5
Attributes: (12/24)
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      35581
    GRIB_typeOfLevel:                         isobaricInhPa
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    GRIB_gridType:                            regular_ll
    ...                                       ...
    GRIB_longitudeOfFirstGridPointInDegrees:  -75.0
    GRIB_longitudeOfLastGridPointInDegrees:   -20.0
    GRIB_missingValue:                        3.4028234663852886e+38
    GRIB_totalNumber:                         0
    GRIB_units:                               m s**-1
    units:                                    m s**-1

In [8]:
tau_ice_ocean = 1_000 * 5.5e-3 * np.sqrt(
    (surface_velocity.u - sea_ice_drift.u) ** 2 + (surface_velocity.v - sea_ice_drift.v) ** 2
) * (surface_velocity.u - sea_ice_drift.u) + 1j * (surface_velocity.v - sea_ice_drift.v)
tau_ice_ocean

<xarray.DataArray (time: 151, latitude: 50, longitude: 161)> Size: 10MB
dask.array<add, shape=(151, 50, 161), dtype=complex64, chunksize=(110, 50, 161), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 1kB 2017-01-03 2017-01-08 ... 2019-01-23
  * latitude   (latitude) float64 400B -74.5 -74.0 -73.5 ... -51.0 -50.5 -50.0
  * longitude  (longitude) float64 1kB -90.0 -89.5 -89.0 ... -11.0 -10.5 -10.0
Attributes:
    cell_methods:   time: mean
    time_avg_info:  average_T1,average_T2,average_DT

In [9]:
tau = (sea_ice_cover.z * tau_ice_ocean) + ((1 - sea_ice_cover.z) * tau_atmosphere_ocean)
tau

<xarray.DataArray (time: 151, latitude: 49, longitude: 110)> Size: 13MB
dask.array<add, shape=(151, 49, 110), dtype=complex128, chunksize=(110, 49, 110), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 1kB 2017-01-03 2017-01-08 ... 2019-01-23
  * latitude   (latitude) float64 392B -74.5 -74.0 -73.5 ... -51.5 -51.0 -50.5
  * longitude  (longitude) float64 880B -75.0 -74.5 -74.0 ... -21.5 -21.0 -20.5
Attributes: (12/32)
    TIFFTAG_IMAGEDESCRIPTION:                 AMSR2 sea ice concentration usi...
    TIFFTAG_SOFTWARE:                         gdal and GMT
    TIFFTAG_ARTIST:                           University of Bremen, Gunnar Sp...
    AREA_OR_POINT:                            Area
    actual_range:                             [0.0, 120.0]
    long_name:                                z
    ...                                       ...
    GRIB_longitudeOfFirstGridPointInDegrees:  -75.0
    GRIB_longitudeOfLastGridPointInDegrees:   -20.0
    GRIB_missingValue:                        3.4028234663852886e+38
    GRIB_totalNumber:                         0
    GRIB_units:                               m s**-1
    units:                                    m s**-1

In [10]:
np.abs(tau).compute().to_zarr("surface_stress_abs.zarr", mode="w")

/albedo/soft/sw/conda-sw/analysis-toolbox-python-04.2026/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
